In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
CATALOG = "olist_dw_project"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
brazil_state_map = {
    "SP": "Sao Paulo", "SC": "Santa Catarina", "MG": "Minas Gerais",
    "PR": "Parana", "RJ": "Rio de Janeiro", "RS": "Rio Grande do Sul",
    "PA": "Para", "GO": "Goias", "ES": "Espirito Santo",
    "BA": "Bahia", "MA": "Maranhao", "MS": "Mato Grosso do Sul",
    "CE": "Ceara", "DF": "Distrito Federal", "RN": "Rio Grande do Norte",
    "PE": "Pernambuco", "MT": "Mato Grosso", "AM": "Amazonas",
    "AP": "Amapa", "AL": "Alagoas", "RO": "Rondonia",
    "PB": "Paraiba", "TO": "Tocantins", "PI": "Piaui",
    "AC": "Acre", "SE": "Sergipe", "RR": "Roraima",
}
mapping_expr = F.create_map([F.lit(x) for pair in brazil_state_map.items() for x in pair])

# customers

In [0]:
df_cust = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

In [0]:
df_cust_silver = (
    df_cust
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
        F.trim(F.col("customer_zip_code_prefix")).alias("customer_zip_code"),
        F.initcap(
            F.regexp_replace(F.trim(F.col("customer_city")), r"\s+", " ")
        ).alias("customer_city"),
        mapping_expr[F.upper(F.trim(F.col("customer_state")))].alias("customer_state"),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.customers"
(
    df_cust_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# sellers

In [0]:
df_sellers = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.sellers")

In [0]:
df_sellers_silver = (
    df_sellers
    .select(
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.trim(F.col("seller_zip_code_prefix")).alias("seller_zip_code"),
        F.initcap(
            F.regexp_replace(F.trim(F.col("seller_city")), r"\s+", " ")
        ).alias("seller_city"),
        mapping_expr[F.upper(F.trim(F.col("seller_state")))].alias("seller_state"),
    )
    .filter(F.col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])  
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.sellers"
(
    df_sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# Order items

In [0]:
df_item = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_items")

In [0]:
df_item_silver = (
    df_item
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("order_item_id").cast(IntegerType()).alias("order_item_id"),
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_ts"),
        F.col("price").cast(DecimalType(10, 2)).alias("price"),
        F.col("freight_value").cast(DecimalType(10, 2)).alias("freight_value"),
    )
    .filter(
        F.col("order_id").isNotNull() 
        & F.col("order_item_id").isNotNull()
        & F.col("product_id").isNotNull()
    )
    .dropDuplicates(["order_id", "order_item_id"])
    .filter((F.col("price") >= 0) & (F.col("freight_value") >= 0))
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.order_items"
(
    df_item_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#order payments

In [0]:
df_pay = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_payments")

In [0]:
df_pay_silver = (
    df_pay.select
    (F.trim(F.col("order_id")).alias("order_id"),
     F.col("payment_sequential").cast(IntegerType()).alias("payment_sequential"),
     F.trim(F.regexp_replace(F.lower(F.col("payment_type")), "_", " ")).alias("payment_type"),
     F.col("payment_installments").cast(IntegerType()).alias("payment_installments"),
     F.col("payment_value").cast(DecimalType(10, 2)).alias("payment_value")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_sequential").isNotNull()
    )
    .dropDuplicates(["order_id", "payment_sequential"])
    .filter(F.col("payment_value") >= 0)
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.order_payments"
(
    df_pay_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# orders

In [0]:
df_order = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders")  

In [0]:
df_order_silver = (
    df_order
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.lower(F.col("order_status"))).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_ts"),
        F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_ts"),
    )
    .filter(
        F.col("order_id").isNotNull() 
        & F.col("customer_id").isNotNull()
        & F.col("order_purchase_ts").isNotNull()
    )
    .dropDuplicates(["order_id"])
)

In [0]:
df_order_silver = (
    df_order_silver
    .withColumn(
        "order_approved_ts",
        F.when(
            F.col("order_delivered_carrier_ts") < F.col("order_approved_ts"),
            F.lit(None)
        ).otherwise(F.col("order_approved_ts"))
    )
    .withColumn(
        "order_delivered_carrier_ts",
        F.when(
            F.col("order_delivered_customer_ts") < F.col("order_delivered_carrier_ts"),
            F.lit(None)
        ).otherwise(F.col("order_delivered_carrier_ts"))
    )
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.orders"
(
    df_order_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# products

In [0]:
df_prod = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.products")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df_products_silver = (
    df_prod
    .select(
        F.trim(F.col("product_id")).alias("product_id"),

        # product_category_name เป็น null ใส่ 'n/a' แทน
        F.coalesce(
            F.trim(
                F.regexp_replace(F.lower(F.col("product_category_name")), "_", " ")
            ),
            F.lit("n/a")
        ).alias("product_category_name"),
        
        # product_name_lenght เป็น null ใส่ 0
        F.coalesce(
            F.col("product_name_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_name_length"),
        
        # product_description_lenght เป็น null ใส่ 0
        F.coalesce(
            F.col("product_description_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_description_length"),
        
        # product_photos_qty เป็น null ใส่ 'n/a' แทน 
        F.coalesce(
            F.col("product_photos_qty").cast(StringType()),
            F.lit("n/a")
        ).alias("product_photos_qty"),
        
        # (<= 0 ให้เป็น null)
        F.when(F.col("product_weight_g") > 0, F.col("product_weight_g").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_weight_g"),
        F.when(F.col("product_length_cm") > 0, F.col("product_length_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_length_cm"),
        F.when(F.col("product_height_cm") > 0, F.col("product_height_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_height_cm"),
        F.when(F.col("product_width_cm") > 0, F.col("product_width_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_width_cm"),
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.products"
(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)